In [13]:
#Lib Imports 
import pandas as pd
import numpy as np
import warnings

# ignorar todos los warnings
warnings.filterwarnings('ignore')


from sklearn.impute import SimpleImputer
from sklearn.preprocessing import RobustScaler
from sklearn_pandas import DataFrameMapper
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_absolute_percentage_error


from tensorflow.keras.models import Sequential, clone_model,save_model
from tensorflow.keras.layers import Input, Dense, Dropout
from tensorflow.keras.callbacks import Callback,EarlyStopping



In [14]:

# cargamos el dataset principal con datos historicos y variables externas
df = pd.read_csv("../data/full_data.csv", parse_dates=["Date"], dayfirst=False)

# ordenamos por fecha
df = df.sort_values("Date").reset_index(drop=True)

# mostramos info general del dataset
print("columnas disponibles:")
print(df.columns.tolist())

print("\nprimeras filas:")
print(df.tail()) #Vemos los ultimos datos

print("\ntipos de datos:")
print(df.dtypes)


columnas disponibles:
['Date', 'Close', 'High', 'Low', 'Open', 'Volume', 'Daily_Change', 'Volatility', 'Pct_Change', 'Volume_Change_pct', 'SMA_7', 'SMA_30', 'Rolling_volatility_30', 'BTC_Close_t-1', 'BTC_Close_t-2', 'BTC_Close_t-3', 'BTC_Close_t-7', 'fng_value', 'fng_classification', 'fng_diff_day', 'fng_SMA_7', 'fng_SMA_30', 'fng_trend', 'Is_Halving_Date', 'Block_reward']

primeras filas:
           Date         Close          High           Low          Open  \
3108 2026-08-06  64262.113281  64934.492188  64098.476562  64595.449219   
3109 2026-08-07  64880.191406  65330.609375  64113.273438  64257.488281   
3110 2026-08-08  64904.687500  65140.480469  64797.113281  64882.546875   
3111 2026-08-09  64844.886719  65401.691406  64677.601562  64906.550781   
3112 2026-08-10  64043.179688  65278.343750  63764.757812  64848.906250   

           Volume  Daily_Change   Volatility  Pct_Change  Volume_Change_pct  \
3108  18529402711   -333.335938   836.015625   -0.005192          -0.213809  

In [15]:
# generacion de targets y analisis de correlacion

# generamos los 7 targets (precio de cierre futuro)
for i in range(1, 8):
    df[f"Close_t+{i}"] = df["Close"].shift(-i)

# eliminamos las filas sin datos completos (las ultimas 7)
df = df.dropna(subset=[f"Close_t+{i}" for i in range(1, 8)]).reset_index(drop=True)

# definimos las features numericas disponibles (todas menos las categoricas o de texto)
available_features = [col for col in df.columns if df[col].dtype != "object" and col not in [f"Close_t+{i}" for i in range(1, 8)]]

# lista de targets
targets = [f"Close_t+{i}" for i in range(1, 8)]


In [16]:
# seleccion de features mas correlacionadas

selected_features = ['Date','Volume','Pct_Change','Volume_Change_pct','Volatility','SMA_7','SMA_30',
             'fng_value','fng_SMA_7','fng_SMA_30','BTC_Close_t-1','BTC_Close_t-2','BTC_Close_t-3','BTC_Close_t-7']


# dejamos solo las columnas seleccionadas y los 7 targets
keep_cols = selected_features + [f"Close_t+{i}" for i in range(1, 8)]
df = df[keep_cols].copy()

print(f"dataset final listo para entrenamiento, con {len(selected_features)} features y 7 targets\n")
print(df.head())


dataset final listo para entrenamiento, con 14 features y 7 targets

        Date       Volume  Pct_Change  Volume_Change_pct   Volatility  SMA_7  \
0 2018-02-01   9959400448         NaN                NaN  1476.519531    NaN   
1 2018-02-02  12726899712   -0.037052           0.277878  1345.790039    NaN   
2 2018-02-03   7263790080    0.038973          -0.429257  1179.120117    NaN   
3 2018-02-04   7073549824   -0.097865          -0.026190  1303.649902    NaN   
4 2018-02-05   9285289984   -0.159688           0.312678  1608.159668    NaN   

   SMA_30  fng_value  fng_SMA_7  fng_SMA_30  ...  BTC_Close_t-2  \
0     NaN       30.0        NaN         NaN  ...            NaN   
1     NaN       15.0        NaN         NaN  ...            NaN   
2     NaN       40.0        NaN         NaN  ...    9170.540039   
3     NaN       24.0        NaN         NaN  ...    8830.750000   
4     NaN       11.0        NaN         NaN  ...    9174.910156   

   BTC_Close_t-3  BTC_Close_t-7    Close_t+1   

In [17]:
# features: todas las columnas numericas excepto los targets y la fecha
targets = [f"Close_t+{i}" for i in range(1, 8)]
features = [c for c in df.columns if c not in targets + ["Date"]]

# diccionarios para guardar X e Y por horizonte
X_dict = {}
Y_dict = {}

for i in range(1, 8):
    tcol = f"Close_t+{i}"
    X_dict[i] = df[features].copy()
    Y_dict[i] = df[[tcol]].copy()

# mostrar shapes para confirmar
print("resumen de shapes por horizonte (i -> X.shape -> y.shape):\n")
for i in range(1, 8):
    print(f"t+{i}: X {X_dict[i].shape} -> Y {Y_dict[i].shape}")

# ejemplo: mostrar las primeras filas del horizonte 1
print("\nprimeras filas - ejemplo horizonte t+1 (X, Y):")
print(X_dict[1].head())
print(Y_dict[1].head())


resumen de shapes por horizonte (i -> X.shape -> y.shape):

t+1: X (3106, 13) -> Y (3106, 1)
t+2: X (3106, 13) -> Y (3106, 1)
t+3: X (3106, 13) -> Y (3106, 1)
t+4: X (3106, 13) -> Y (3106, 1)
t+5: X (3106, 13) -> Y (3106, 1)
t+6: X (3106, 13) -> Y (3106, 1)
t+7: X (3106, 13) -> Y (3106, 1)

primeras filas - ejemplo horizonte t+1 (X, Y):
        Volume  Pct_Change  Volume_Change_pct   Volatility  SMA_7  SMA_30  \
0   9959400448         NaN                NaN  1476.519531    NaN     NaN   
1  12726899712   -0.037052           0.277878  1345.790039    NaN     NaN   
2   7263790080    0.038973          -0.429257  1179.120117    NaN     NaN   
3   7073549824   -0.097865          -0.026190  1303.649902    NaN     NaN   
4   9285289984   -0.159688           0.312678  1608.159668    NaN     NaN   

   fng_value  fng_SMA_7  fng_SMA_30  BTC_Close_t-1  BTC_Close_t-2  \
0       30.0        NaN         NaN            NaN            NaN   
1       15.0        NaN         NaN    9170.540039          

In [18]:

#targets y features
targets = [f"Close_t+{i}" for i in range(1, 8)]  # 7 targets, 7 modelos
features = [col for col in df.columns if col not in targets + ['Date']]  # usamos todas las features excepto targets y date

#columnas a escalar
cols_scaler = features  # todas las numericas

#crear mapper
# mapper aplica: primero imputa valores nulos con la mediana, luego escala con robust scaler
mapper = DataFrameMapper([
    (cols_scaler, [SimpleImputer(strategy='median'), RobustScaler()])
], input_df=True, df_out=True)

#definir modelos
models_dict = {
    "LinearRegression": LinearRegression(),
    "KNN": KNeighborsRegressor(n_neighbors=15),
    "DecisionTree": DecisionTreeRegressor(max_depth=None, random_state=42),
    "RandomForest": RandomForestRegressor(n_estimators=100, random_state=42)
}

#pipelines vacios por cada target
pipelines = {}
for target in targets:
    # aca solo definimos el pipeline base, luego le metemos cada modelo de la lista, 28 pipelines
    pipelines[target] = {}
    
    for model_name, model in models_dict.items():
        pipeline = Pipeline([
            ('mapper', mapper),
            ('model', model)
        ])
        pipelines[target][model_name] = pipeline


In [19]:

# definimos tamaño de test final
test_size = 0.1
n_test = int(len(df) * test_size)

# separamos test final
train_val_df = df.iloc[:-n_test].reset_index(drop=True)
test_df = df.iloc[-n_test:].reset_index(drop=True)

n_splits = 7
tscv = TimeSeriesSplit(n_splits=n_splits)

# diccionarios para guardar índices por fold y por target
folds_idx = {target: [] for target in targets}

print(f"División en {n_splits} folds (train/val) por fechas:\n")
for target in targets:
    print(f" Target: {target}")
    X = train_val_df[features]
    y = train_val_df[[target]]

    for fold, (train_idx, val_idx) in enumerate(tscv.split(X)):
        # guardar indices
        folds_idx[target].append((train_idx, val_idx))

        # fechas para mostrar
        start_train = train_val_df.iloc[train_idx[0]]['Date']
        end_train   = train_val_df.iloc[train_idx[-1]]['Date']
        start_val   = train_val_df.iloc[val_idx[0]]['Date']
        end_val     = train_val_df.iloc[val_idx[-1]]['Date']

        print(f"Fold {fold+1}:")
        print(f"  Train: {start_train.date()} -> {end_train.date()} ({len(train_idx)} filas)")
        print(f"  Val:   {start_val.date()} -> {end_val.date()} ({len(val_idx)} filas)\n")

# mostrar tamaño test final
print(f"Test final: {test_df['Date'].min().date()} -> {test_df['Date'].max().date()} ({len(test_df)} filas)")


División en 7 folds (train/val) por fechas:

 Target: Close_t+1
Fold 1:
  Train: 2018-02-01 -> 2019-01-19 (353 filas)
  Val:   2019-01-20 -> 2020-01-03 (349 filas)

Fold 2:
  Train: 2018-02-01 -> 2020-01-03 (702 filas)
  Val:   2020-01-04 -> 2020-12-17 (349 filas)

Fold 3:
  Train: 2018-02-01 -> 2020-12-17 (1051 filas)
  Val:   2020-12-18 -> 2021-12-01 (349 filas)

Fold 4:
  Train: 2018-02-01 -> 2021-12-01 (1400 filas)
  Val:   2021-12-02 -> 2022-11-15 (349 filas)

Fold 5:
  Train: 2018-02-01 -> 2022-11-15 (1749 filas)
  Val:   2022-11-16 -> 2023-10-30 (349 filas)

Fold 6:
  Train: 2018-02-01 -> 2023-10-30 (2098 filas)
  Val:   2023-10-31 -> 2024-10-13 (349 filas)

Fold 7:
  Train: 2018-02-01 -> 2024-10-13 (2447 filas)
  Val:   2024-10-14 -> 2025-09-27 (349 filas)

 Target: Close_t+2
Fold 1:
  Train: 2018-02-01 -> 2019-01-19 (353 filas)
  Val:   2019-01-20 -> 2020-01-03 (349 filas)

Fold 2:
  Train: 2018-02-01 -> 2020-01-03 (702 filas)
  Val:   2020-01-04 -> 2020-12-17 (349 filas)

Fol

In [20]:

n_splits = 7
tscv = TimeSeriesSplit(n_splits=n_splits)

# diccionario para resultados
cv_results = {}  # {target: {model_name: [fold_results]}}

for target in targets:
    print(f"\n Target: {target}")
    X = df[features]
    y = df[[target]]

    cv_results[target] = {}

    for model_name, pipeline in pipelines[target].items():
        print(f"\n Modelo: {model_name}")
        fold_results = []

        for fold, (train_idx, val_idx) in enumerate(tscv.split(X)):
            X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
            y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

            # entrenar
            pipeline.fit(X_train, y_train)

            # predecir
            y_pred_train = pipeline.predict(X_train)
            y_pred_val   = pipeline.predict(X_val)

            # métricas
            mae_train = mean_absolute_error(y_train, y_pred_train)
            rmse_train = mean_squared_error(y_train, y_pred_train)**0.5

            mae_val = mean_absolute_error(y_val, y_pred_val)
            rmse_val = mean_squared_error(y_val, y_pred_val)**0.5

            fold_results.append({
                "fold": fold+1,
                "MAE_train": mae_train, "RMSE_train": rmse_train,
                "MAE_val": mae_val, "RMSE_val": rmse_val
            })

            print(f"Fold {fold+1}: Train MAE {mae_train:.2f}, RMSE {rmse_train:.2f} | "
                  f"Val MAE {mae_val:.2f}, RMSE {rmse_val:.2f}")

        cv_results[target][model_name] = fold_results



 Target: Close_t+1

 Modelo: LinearRegression
Fold 1: Train MAE 194.87, RMSE 293.91 | Val MAE 289.62, RMSE 430.32
Fold 2: Train MAE 211.24, RMSE 331.11 | Val MAE 792.24, RMSE 1442.95
Fold 3: Train MAE 407.13, RMSE 783.41 | Val MAE 1747.14, RMSE 2268.70
Fold 4: Train MAE 740.56, RMSE 1218.06 | Val MAE 491.34, RMSE 756.23
Fold 5: Train MAE 696.00, RMSE 1142.30 | Val MAE 885.96, RMSE 1407.76
Fold 6: Train MAE 739.74, RMSE 1209.59 | Val MAE 1901.82, RMSE 2544.66
Fold 7: Train MAE 919.05, RMSE 1485.38 | Val MAE 1685.43, RMSE 2282.19

 Modelo: KNN
Fold 1: Train MAE 324.58, RMSE 490.54 | Val MAE 935.91, RMSE 1137.33
Fold 2: Train MAE 327.36, RMSE 498.93 | Val MAE 11037.15, RMSE 19398.99
Fold 3: Train MAE 548.90, RMSE 1033.72 | Val MAE 3735.29, RMSE 4566.10
Fold 4: Train MAE 1485.97, RMSE 2219.50 | Val MAE 8833.81, RMSE 9702.11
Fold 5: Train MAE 1330.92, RMSE 2019.31 | Val MAE 4353.31, RMSE 5473.92
Fold 6: Train MAE 1544.90, RMSE 2318.91 | Val MAE 21536.95, RMSE 25567.42
Fold 7: Train MAE 195

In [21]:

# features y targets ya definidos

# diccionario para guardar pesos por fold/epoch
model_weights_by_horizon = {}

# Callback para guardar pesos
class OurCustomCallback(Callback):
    def __init__(self, horizon):
        super().__init__()
        self.horizon = horizon
        
    def on_epoch_end(self, epoch, logs=None):
        import copy
        if self.horizon not in model_weights_by_horizon:
            model_weights_by_horizon[self.horizon] = {}
        model_weights_by_horizon[self.horizon][epoch] = copy.deepcopy(self.model.get_weights())

# función para crear un modelo base
def create_mlp_model(input_dim):
    model = Sequential([
        Input(shape=(input_dim,)),
        Dense(128, activation='relu'),
        Dropout(0.2),
        Dense(128, activation='relu'),
        Dense(64, activation='relu'),
        Dense(1, activation='linear')  # salida para un solo target
    ])
    model.compile(optimizer='adam', loss='mse', metrics=['mae','mse'])
    return model


In [23]:

# para guardar resultados
cv_results_nn = {}

# escalador
scaler = RobustScaler()

# early stopping para evitar sobreajuste
early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

for i, target in enumerate(targets, start=1):
    print(f"\n Entrenando modelo NN para {target}")
    cv_results_nn[target] = []

    X = train_val_df[features].values
    y = train_val_df[[target]].values

    for fold, (train_idx, val_idx) in enumerate(folds_idx[target]):
        print(f"\n Fold {fold+1}/{len(folds_idx[target])}")

        # separar sets
        X_train, X_val = X[train_idx], X[val_idx]
        y_train, y_val = y[train_idx], y[val_idx]

        # escalar usando solo el train
        X_train_scaled = scaler.fit_transform(X_train)
        X_val_scaled = scaler.transform(X_val)

        # crear nuevo modelo
        model = create_mlp_model(X_train_scaled.shape[1])

        # callback personalizado (opcional)
        callback = OurCustomCallback(horizon=i)

        # entrenar
        history = model.fit(
            X_train_scaled, y_train,
            validation_data=(X_val_scaled, y_val),
            epochs=150,
            batch_size=32,
            verbose=0
        )

        # predicciones
        y_pred_train = model.predict(X_train_scaled)
        y_pred_val = model.predict(X_val_scaled)

        # métricas
        mae_train = mean_absolute_error(y_train, y_pred_train)
        rmse_train = mean_squared_error(y_train, y_pred_train) ** 0.5
        mape_train = mean_absolute_percentage_error(y_train, y_pred_train)

        mae_val = mean_absolute_error(y_val, y_pred_val)
        rmse_val = mean_squared_error(y_val, y_pred_val) ** 0.5
        mape_val = mean_absolute_percentage_error(y_val, y_pred_val)

        print(
            f"Fold {fold+1}: "
            f"Train -> MAE={mae_train:.4f}, RMSE={rmse_train:.4f} | "
            f"Val -> MAE={mae_val:.4f}, RMSE={rmse_val:.4f}"
        )

        # guardar resultados
        cv_results_nn[target].append({
            'fold': fold+1,
            'MAE_train': mae_train,
            'RMSE_train': rmse_train,
            'MAPE_train': mape_train,
            'MAE_val': mae_val,
            'RMSE_val': rmse_val,
            'MAPE_val': mape_val
        })

# promedio final por target
print("\n Resultados Promedio por Horizonte")
for target in targets:
    mae_train_mean = np.mean([r['MAE_train'] for r in cv_results_nn[target]])
    mae_val_mean = np.mean([r['MAE_val'] for r in cv_results_nn[target]])
    rmse_train_mean = np.mean([r['RMSE_train'] for r in cv_results_nn[target]])
    rmse_val_mean = np.mean([r['RMSE_val'] for r in cv_results_nn[target]])

    print(
        f"{target}: "
        f"Train -> MAE={mae_train_mean:.4f}, RMSE={rmse_train_mean:.4f} | "
        f"Val -> MAE={mae_val_mean:.4f}, RMSE={rmse_val_mean:.4f}"
    )



 Entrenando modelo NN para Close_t+1

 Fold 1/7
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step
11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step 
Fold 1: Train -> MAE=1398.2530, RMSE=1864.8491 | Val -> MAE=2332.5012, RMSE=2663.2494

 Fold 2/7
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step
11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step 
Fold 2: Train -> MAE=1830.4111, RMSE=2258.0022 | Val -> MAE=3576.5337, RMSE=4766.5990

 Fold 3/7
33/33 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step
11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step 
Fold 3: Train -> MAE=2286.7522, RMSE=3110.7284 | Val -> MAE=38150.7278, RMSE=39652.2851

 Fold 4/7
44/44 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step 
Fold 4: Train -> MAE=14346.2824, RMSE=17571.1148 | Val -> MAE=13522.1415, RMSE=17273.7145

 Fold 5/7
55/55 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step 
Fold 5: Train -> MAE=14803.6205, RMSE=17309.1362 | Val -> MAE=6046.5026, RMSE=6671.2505

 Fold 6/7
66/66 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
11/11 ━━━━━━━━━━━━━━━━━━

In [24]:
import os
import joblib  # para guardar el scaler

# directorio para guardar modelos
output_dir = "./models_final"
os.makedirs(output_dir, exist_ok=True)

# diccionario para guardar modelos y resultados
final_models = {}
final_metrics = {}

# iterar sobre los 7 horizontes
for i, target in enumerate(targets, start=1):
    print(f"\n Entrenando modelo final para {target}")
    
    # X e y completos (train + val)
    X = train_val_df[features].values
    y = train_val_df[[target]].values

    # escalar
    scaler_final = RobustScaler()
    X_scaled = scaler_final.fit_transform(X)
    
    # crear modelo
    model = create_mlp_model(X_scaled.shape[1])

    # entrenar
    history = model.fit(
        X_scaled, y,
        epochs=200,
        batch_size=32,
        verbose=0
    )

    # predicciones sobre todo el dataset
    y_pred = model.predict(X_scaled)

    # calcular metricas
    mae = mean_absolute_error(y, y_pred)
    rmse = mean_squared_error(y, y_pred) ** 0.5
    mape = mean_absolute_percentage_error(y, y_pred)

    print(f"{target} -> MAE: {mae:.4f}, RMSE: {rmse:.4f}, MAPE: {mape:.4f}")

    # guardar modelo en h5
    model_path = os.path.join(output_dir, f"mlp_{target}.h5")
    save_model(model, model_path)
    print(f"modelo guardado en {model_path}")

    # guardar scaler asociado a este horizonte
    scaler_path = os.path.join(output_dir, f"scaler_{target}.pkl")
    joblib.dump(scaler_final, scaler_path)
    print(f"scaler guardado en {scaler_path}")

    # guardar en diccionario para uso inmediato
    final_models[target] = {
        "model": model,
        "scaler": scaler_final
    }

    # guardar metricas
    final_metrics[target] = {
        "MAE": mae,
        "RMSE": rmse,
        "MAPE": mape
    }

print("\n Todos los modelos finales, scalers y metricas guardados")
for target, met in final_metrics.items():
    print(f"{target}: MAE={met['MAE']:.4f}, RMSE={met['RMSE']:.4f}, MAPE={met['MAPE']:.4f}")



 Entrenando modelo final para Close_t+1
88/88 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step


Close_t+1 -> MAE: 24841.5837, RMSE: 30600.2209, MAPE: 1.7160
modelo guardado en ./models_final\mlp_Close_t+1.h5
scaler guardado en ./models_final\scaler_Close_t+1.pkl

 Entrenando modelo final para Close_t+2
88/88 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step


Close_t+2 -> MAE: 24882.8961, RMSE: 30632.3797, MAPE: 1.7214
modelo guardado en ./models_final\mlp_Close_t+2.h5
scaler guardado en ./models_final\scaler_Close_t+2.pkl

 Entrenando modelo final para Close_t+3
88/88 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step


Close_t+3 -> MAE: 24900.2393, RMSE: 30664.3782, MAPE: 1.7203
modelo guardado en ./models_final\mlp_Close_t+3.h5
scaler guardado en ./models_final\scaler_Close_t+3.pkl

 Entrenando modelo final para Close_t+4
88/88 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step


Close_t+4 -> MAE: 24947.3194, RMSE: 30700.3300, MAPE: 1.7268
modelo guardado en ./models_final\mlp_Close_t+4.h5
scaler guardado en ./models_final\scaler_Close_t+4.pkl

 Entrenando modelo final para Close_t+5
88/88 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
Close_t+5 -> MAE: 24951.1928, RMSE: 30737.7514, MAPE: 1.7211


modelo guardado en ./models_final\mlp_Close_t+5.h5
scaler guardado en ./models_final\scaler_Close_t+5.pkl

 Entrenando modelo final para Close_t+6
88/88 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step


Close_t+6 -> MAE: 24997.9785, RMSE: 30776.8254, MAPE: 1.7272
modelo guardado en ./models_final\mlp_Close_t+6.h5
scaler guardado en ./models_final\scaler_Close_t+6.pkl

 Entrenando modelo final para Close_t+7
88/88 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step


Close_t+7 -> MAE: 25015.3645, RMSE: 30815.9938, MAPE: 1.7251
modelo guardado en ./models_final\mlp_Close_t+7.h5
scaler guardado en ./models_final\scaler_Close_t+7.pkl

 Todos los modelos finales, scalers y metricas guardados
Close_t+1: MAE=24841.5837, RMSE=30600.2209, MAPE=1.7160
Close_t+2: MAE=24882.8961, RMSE=30632.3797, MAPE=1.7214
Close_t+3: MAE=24900.2393, RMSE=30664.3782, MAPE=1.7203
Close_t+4: MAE=24947.3194, RMSE=30700.3300, MAPE=1.7268
Close_t+5: MAE=24951.1928, RMSE=30737.7514, MAPE=1.7211
Close_t+6: MAE=24997.9785, RMSE=30776.8254, MAPE=1.7272
Close_t+7: MAE=25015.3645, RMSE=30815.9938, MAPE=1.7251


In [ ]:


# diccionario para guardar métricas de test
test_metrics = {}

for target in targets:
    print(f"\n Evaluando modelo en test para {target}")

    # obtener modelo y scaler desde final_models
    model = final_models[target]["model"]
    scaler = final_models[target]["scaler"]

    # preparar X e y de test
    X_test = test_df[features].values
    y_test = test_df[[target]].values

    # escalar usando el scaler
    X_test_scaled = scaler.transform(X_test)

    # predecir
    y_pred_test = model.predict(X_test_scaled)

    # calcular métricas
    mae = mean_absolute_error(y_test, y_pred_test)
    rmse = mean_squared_error(y_test, y_pred_test) ** 0.5
    mape = mean_absolute_percentage_error(y_test, y_pred_test)

    print(f"{target} -> MAE: {mae:.4f}, RMSE: {rmse:.4f}, MAPE: {mape:.4f}")

    # guardar métricas
    test_metrics[target] = {
        "MAE": mae,
        "RMSE": rmse,
        "MAPE": mape
    }

print("\n métricas finales sobre test")
for target, met in test_metrics.items():
    print(f"{target}: MAE={met['MAE']:.4f}, RMSE={met['RMSE']:.4f}, MAPE={met['MAPE']:.4f}")



 Evaluando modelo en test para Close_t+1
9/9 [==============================] - 0s 750us/step
Close_t+1 -> MAE: 1793.9860, RMSE: 2337.0098, MAPE: 0.0179

 Evaluando modelo en test para Close_t+2
9/9 [==============================] - 0s 875us/step
Close_t+2 -> MAE: 2306.1575, RMSE: 3061.1353, MAPE: 0.0228

 Evaluando modelo en test para Close_t+3
9/9 [==============================] - 0s 875us/step
Close_t+3 -> MAE: 2768.5951, RMSE: 3599.8844, MAPE: 0.0277

 Evaluando modelo en test para Close_t+4
9/9 [==============================] - 0s 1000us/step
Close_t+4 -> MAE: 2988.8505, RMSE: 3910.2352, MAPE: 0.0294

 Evaluando modelo en test para Close_t+5
9/9 [==============================] - 0s 875us/step
Close_t+5 -> MAE: 3366.2247, RMSE: 4326.5930, MAPE: 0.0331

 Evaluando modelo en test para Close_t+6
9/9 [==============================] - 0s 875us/step
Close_t+6 -> MAE: 3630.1895, RMSE: 4634.5240, MAPE: 0.0357

 Evaluando modelo en test para Close_t+7
9/9 [============================